# 计算机视觉实践课 · Python 图像处理

> 配套练习：[`02_练习题.ipynb`](./02_练习题.ipynb)　|　参考答案：[`03_练习题答案.ipynb`](./03_练习题答案.ipynb)

本 Notebook 是一份**可以直接从上到下依次运行**的实践课讲义，覆盖图像处理最常用的五个 Python 工具：

| 章节 | 工具 | 你将学会 |
| :--- | :--- | :--- |
| 1.1 | Anaconda / pip | 搭建可复现的 Python 环境，检查依赖版本 |
| 1.2 | **OpenCV** (`cv2`) | 读写图像（含中文路径）、颜色空间转换、缩放、绘图 |
| 1.3 | **Pillow** (`PIL`) | 模式转换、缩放旋转、内置滤镜、与 ndarray / cv2 互转 |
| 1.4 | **SciPy** (`scipy.ndimage`) | 均值/高斯模糊、卷积与相关的区别、去噪实践与 PSNR 评估 |
| 1.5 | **NumPy** | 灰度变换、手写插值缩放、Unsharp Masking 锐化、多帧平均降噪 |
| 1.6 | **Matplotlib** | `imshow` / `subplot` / `hist` / `colorbar` / 保存图像 |

**使用建议**

1. 菜单 `Kernel → Restart & Run All`，一次性跑完全部内容；
2. 每个代码单元后面都附有「运行结果说明」，先自己预测输出，再对照运行结果；
3. 所有示例图像**默认从 Notebook 同目录的 `images/` 文件夹加载**（主图为经典测试图 **Lenna**，当前 640×640，代码按实际尺寸自适应）；即使 `images/` 缺失或损坏，代码也会**自动降级为 NumPy 现场合成图像**，因此本课程包拷到任何一台机器上都能跑通，绝不因缺图中断。

---

## 0. 课前准备：图像素材与公共工具

本课程的所有示例图像存放在 Notebook 同目录下的 **`images/`** 文件夹中（主图为经典测试图 **Lenna**，当前 640×640；代码按文件实际尺寸自适应）。
即使 `images/` 缺失或损坏，代码也会**优雅降级**为 NumPy 现场合成图像，绝不因缺图中断。

下面 4 个代码单元定义了全课复用的公共工具（路径解析、合成图像、噪声/指标、显示函数），**三份 Notebook 完全一致**。

In [ ]:
# ============================================================
# 公共工具 (1/4)：路径解析 + 本地图像加载
# 依赖：numpy、pillow
# ============================================================
import os
import numpy as np
import glob as _glob

# def _resolve_notebook_dir():
#     """返回 .ipynb 所在目录的绝对路径（尽可能稳健）。"""
#     import glob as _glob
#     # 方式 ：从 cwd 向上最多 4 级查找含 .ipynb 的目录
#     d = os.path.abspath(os.getcwd())
# #     for _ in range(5):
# #         if _glob.glob(os.path.join(d, "*.ipynb")):
# #             return d
# #         parent = os.path.dirname(d)
# #         if parent == d:
# #             break
# #         d = parent
#     return os.path.abspath(os.getcwd())


NOTEBOOK_DIR = os.path.abspath(os.getcwd())
IMAGES_DIR = os.path.join(NOTEBOOK_DIR, "images")

# 主示例图路径（Lenna；实际尺寸以 images/ 内文件为准，当前为 640×640）
LENNA_COLOR_PATH = os.path.join(IMAGES_DIR, "lenna.png")
LENNA_GRAY_PATH = os.path.join(IMAGES_DIR, "lenna_gray.png")


def center_crop_square(img):
    """中心裁剪为正方形（边长 = min(H, W)）；已是正方形则原样返回。

    供「需要方形输入」的环节使用：无论素材长宽比如何变化，
    都能拿到居中的方形区域，避免错位或形状报错。
    """
    img = np.asarray(img)
    h, w = img.shape[:2]
    if h == w:
        return img
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return img[y0:y0 + side, x0:x0 + side]


def load_lenna_color(size=None, verbose=True):
    """加载 Lenna 彩色图。优先读 images/lenna.png，失败则合成替代图。

    size=None 时保持素材**原始尺寸**（并中心裁剪为方形）；
    仅当显式传入 size 时才缩放到 (size, size)。
    """
    if os.path.isfile(LENNA_COLOR_PATH):
        try:
            from PIL import Image
            im = Image.open(LENNA_COLOR_PATH).convert("RGB")
            arr = center_crop_square(np.asarray(im, dtype=np.uint8))
            if size:
                arr = np.asarray(Image.fromarray(arr).resize((size, size), Image.LANCZOS),
                                 dtype=np.uint8)
            h, w = arr.shape[:2]
            if verbose:
                print(f"[素材] Lenna 彩色图：{LENNA_COLOR_PATH} ({w}×{h})")
            return arr, f"Lenna 彩色 ({w}×{h})"
        except Exception as e:
            if verbose:
                print(f"[素材] Lenna 读取失败（{e}），降级为合成图像")
    else:
        if verbose:
            print(f"[素材] 未找到 {LENNA_COLOR_PATH}，降级为合成图像")
    # 降级：合成一张有丰富纹理的测试图
    if verbose:
        print("[素材] 使用 NumPy 合成图像（课程自包含，可离线运行）")
    fall = make_sunset_rgb(size or 480)
    return fall, f"NumPy 合成图像（日落场景 {fall.shape[1]}×{fall.shape[0]}）"


def load_lenna_gray(size=None, verbose=False):
    """加载 Lenna 灰度图。优先读 images/lenna_gray.png，否则从彩色转换。"""
    if os.path.isfile(LENNA_GRAY_PATH):
        try:
            from PIL import Image
            im = Image.open(LENNA_GRAY_PATH).convert("L")
            arr = center_crop_square(np.asarray(im, dtype=np.uint8))
            if size:
                arr = np.asarray(Image.fromarray(arr).resize((size, size), Image.LANCZOS),
                                 dtype=np.uint8)
            h, w = arr.shape[:2]
            return arr, f"Lenna 灰度 ({w}×{h})"
        except Exception:
            pass
    # 从彩色转灰度
    rgb, src = load_lenna_color(size, verbose=verbose)
    gray = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2])
    return np.clip(gray, 0, 255).astype(np.uint8), src.replace("彩色", "灰度")


print("Notebook 目录     :", NOTEBOOK_DIR)
print("图像素材目录      :", IMAGES_DIR, "存在" if os.path.isdir(IMAGES_DIR) else "不存在")
print("Lenna 彩色        :", "✓" if os.path.isfile(LENNA_COLOR_PATH) else "✗ (将降级合成)")


**运行结果说明**：上面的单元会打印诊断信息，确认 Notebook 目录与 Lenna 图像是否就位。

In [ ]:
# ============================================================
# 公共工具 (2/4)：合成图像（降级备用）
# ============================================================


def make_sunset_rgb(size=480):
    """合成一张日落风景图 (size, size, 3) uint8，作为 Lenna 不可用时的降级。"""
    h = w = size
    img = np.zeros((h, w, 3), np.uint8)
    y_coords = np.linspace(0, 1, h)[:, None]
    x_coords = np.linspace(0, 1, w)[None, :]
    # 天空渐变
    for c in range(3):
        sky = np.linspace([40, 20, 80][c], [255, 140, 50][c], h // 2)[:, None]
        img[:h//2, :, c] = np.broadcast_to(sky, (h//2, w))
    # 太阳光晕
    cx, cy, r = w * 0.6, h * 0.35, size * 0.08
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    glow = np.clip(1.0 - dist / (r * 4), 0, 1) ** 2
    img[..., 0] = np.clip(img[..., 0] + glow * 200, 0, 255).astype(np.uint8)
    img[..., 1] = np.clip(img[..., 1] + glow * 150, 0, 255).astype(np.uint8)
    # 山峦
    ridge = (h * 0.5 + 30 * np.sin(x_coords * 6 * np.pi) + 15 * np.sin(x_coords * 14 * np.pi)).astype(int)
    for col in range(w):
        r_val = np.clip(ridge[0, col], h//2, h-1)
        img[int(r_val):, col, :] = [30, 60, 30]
    # 水面波纹
    water_start = int(h * 0.72)
    for row in range(water_start, h):
        frac = (row - water_start) / max(h - water_start, 1)
        wave = (10 * np.sin(np.arange(w) * 0.15 + frac * 8)).astype(int)
        img[row, :, 0] = np.clip(20 + frac * 30 + wave, 0, 255).astype(np.uint8)
        img[row, :, 2] = np.clip(60 + frac * 50 + wave, 0, 255).astype(np.uint8)
    return img


def make_test_gray(size=480):
    """合成四象限灰度试纸 (size, size) uint8。"""
    img = np.zeros((size, size), np.uint8)
    half = size // 2
    # 左上：水平渐变
    img[:half, :half] = np.tile(np.linspace(0, 255, half, dtype=np.uint8), (half, 1))
    # 右上：垂直渐变
    img[:half, half:] = np.tile(np.linspace(0, 255, half, dtype=np.uint8)[:, None], (1, size - half))
    # 左下：棋盘格
    block = max(size // 16, 4)
    for r in range(half, size):
        for c in range(half):
            img[r, c] = 255 if ((r - half) // block + c // block) % 2 == 0 else 0
    # 右下：同心圆
    cy, cx = size * 3 // 4, size * 3 // 4
    yy, xx = np.ogrid[half:size, half:size]
    dist = np.sqrt((yy - cy)**2 + (xx - cx)**2)
    img[half:, half:] = ((np.sin(dist * 0.3) + 1) * 127).astype(np.uint8)
    return img


def make_color_bars(size=360):
    """合成彩条图 (size, size, 3) uint8。"""
    colors = [(255,255,255),(255,255,0),(0,255,255),(0,255,0),
              (255,0,255),(255,0,0),(0,0,255),(0,0,0)]
    bar_w = size // len(colors)
    img = np.zeros((size, size, 3), np.uint8)
    for i, c in enumerate(colors):
        img[:, i*bar_w:(i+1)*bar_w] = c
    return img


print("合成函数已定义：make_sunset_rgb / make_test_gray / make_color_bars")
_t = make_test_gray(64)
print(f"自检：make_test_gray(64) -> shape={_t.shape}, dtype={_t.dtype}")
del _t


**运行结果说明**：三个合成函数都返回 `uint8` 数组，形状符合 `(H, W)` / `(H, W, 3)` 约定。

In [ ]:
# ============================================================
# 公共工具 (3/4)：噪声生成 + 质量指标
# ============================================================


def add_gaussian_noise(img, sigma=20.0, seed=0):
    """加加性高斯噪声：g(x,y) = f(x,y) + n,  n ~ N(0, sigma^2)"""
    rng = np.random.default_rng(seed)
    noisy = img.astype(np.float64) + rng.normal(0.0, sigma, img.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)


def add_salt_pepper_noise(img, amount=0.02, seed=0):
    """加椒盐噪声：随机把若干像素置为 0（胡椒）或 255（盐）。"""
    rng = np.random.default_rng(seed)
    out = img.copy()
    h, w = out.shape[:2]
    n = max(int(h * w * amount / 2.0), 1)
    for value in (255, 0):
        ys = rng.integers(0, h, n)
        xs = rng.integers(0, w, n)
        if out.ndim == 3:
            out[ys, xs, :] = value
        else:
            out[ys, xs] = value
    return out


def make_noisy_frames(img, n_frames=32, sigma=25.0, seed=100):
    """生成 n_frames 张含独立高斯噪声的图。"""
    return [add_gaussian_noise(img, sigma=sigma, seed=seed + i) for i in range(n_frames)]


def mse(a, b):
    """均方误差 MSE = mean((a-b)^2)。"""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"形状不一致：{a.shape} vs {b.shape}")
    return float(np.mean((a - b) ** 2))


def psnr(a, b, peak=255.0):
    """峰值信噪比 PSNR (dB)。完全相同返回 inf。"""
    m = mse(a, b)
    if m <= 1e-12:
        return float("inf")
    return float(10.0 * np.log10(peak ** 2 / m))


print("噪声与指标函数已定义：add_gaussian_noise / add_salt_pepper_noise / "
      "make_noisy_frames / mse / psnr")
_g = make_test_gray(128)
print(f"自检：原图 vs 原图            PSNR = {psnr(_g, _g):.4g} dB (应为 inf)")
print(f"自检：原图 vs 高斯噪声(σ=20)  PSNR = {psnr(_g, add_gaussian_noise(_g, 20, 0)):.2f} dB")
del _g


In [ ]:
# ============================================================
# 公共工具 (4/4)：统一图像入口 + Matplotlib 显示/中文字体
# ============================================================


def setup_matplotlib(candidates=None):
    """配置 Matplotlib 中文字体与负号显示。"""
    import matplotlib
    from matplotlib import font_manager
    cands = candidates or ["PingFang SC", "Hiragino Sans GB", "Heiti SC", "STHeiti",
                           "Songti SC", "Arial Unicode MS", "Microsoft YaHei", "SimHei",
                           "Noto Sans CJK SC", "Source Han Sans SC", "WenQuanYi Zen Hei"]
    available = {f.name for f in font_manager.fontManager.ttflist}
    chosen = next((c for c in cands if c in available), None)
    if chosen is not None:
        cur = list(matplotlib.rcParams["font.sans-serif"])
        matplotlib.rcParams["font.sans-serif"] = [chosen] + [f for f in cur if f != chosen]
    matplotlib.rcParams["font.family"] = "sans-serif"
    matplotlib.rcParams["axes.unicode_minus"] = False
    return chosen


def show_images(images, titles=None, cols=None, figsize=None, cmap=None,
                colorbar=False, suptitle=None, save_path=None, dpi=150):
    """并排对比显示多幅图像。"""
    import matplotlib.pyplot as plt
    n = len(images)
    if cols is None:
        cols = n
    rows = (n + cols - 1) // cols
    if figsize is None:
        figsize = (4.2 * cols, 4.2 * rows + 0.6)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for idx, img in enumerate(images):
        ax = axes[idx // cols, idx % cols]
        kw = {}
        if img.ndim == 2:
            kw["cmap"] = cmap or "gray"
        im = ax.imshow(img, **kw)
        if colorbar:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if titles and idx < len(titles):
            ax.set_title(titles[idx], fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    # 隐藏多余子图
    for idx in range(n, rows * cols):
        axes[idx // cols, idx % cols].set_visible(False)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, y=0.995)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"[show_images] 已保存: {save_path}")
    plt.show()
    return fig


def to_uint8(arr):
    """把 float 数组 clip 到 [0,255] 并转 uint8。"""
    return np.clip(arr, 0, 255).astype(np.uint8)


# ---- 全局初始化 ----
import matplotlib
import matplotlib.pyplot as plt

_font = setup_matplotlib()
print("Matplotlib 中文字体 :", _font or "（未找到中文字体，标题可能显示方框）")
print("Matplotlib 后端     :", matplotlib.get_backend())

WORK_DIR = os.path.join(NOTEBOOK_DIR, "_output")
os.makedirs(WORK_DIR, exist_ok=True)
print("输出目录 WORK_DIR   :", WORK_DIR)

# 加载主演示图像（Lenna）
RGB_IMG, IMG_SRC = load_lenna_color()
GRAY_IMG, _ = load_lenna_gray()

print(f"\n主演示图像 RGB_IMG : shape={RGB_IMG.shape}, dtype={RGB_IMG.dtype}, 来源={IMG_SRC}")
print(f"主演示图像 GRAY_IMG: shape={GRAY_IMG.shape}, dtype={GRAY_IMG.dtype}")

show_images([RGB_IMG, GRAY_IMG],
            titles=["Lenna 彩色", "Lenna 灰度"],
            cols=2, suptitle="素材总览")


**运行结果说明**：最后一行会显示一张 1×4 的素材总览图。

## 1.1 Python 环境搭建

### 1.1.1 为什么推荐 Anaconda？

图像处理依赖大量带**编译好的 C/C++ 底层**的库（OpenCV、NumPy、SciPy）。直接用系统 Python + `pip` 安装，经常会遇到「缺少编译工具链」「版本互相冲突」等问题。**Anaconda / Miniconda** 提供：

- **预编译二进制包**：`conda install` 直接下载编译好的包，无需本地编译；
- **环境隔离**：每个项目一个独立环境，互不干扰（比如课程用 Python 3.11，别的项目用 3.9）；
- **跨平台一致**：Windows / macOS / Linux 命令相同。

> 💡 课堂建议：安装体积更小的 **Miniconda**（约 100 MB），需要的大包再按需 `conda install` / `pip install`。

### 1.1.2 安装与环境管理（在终端中执行，**不是**在 Notebook 里）

```bash
# ---------- 1) 安装 Miniconda 后，初始化 shell ----------
conda init zsh        # macOS / Linux 默认 zsh 或 bash
conda init powershell # Windows

# ---------- 2) 创建本课程专用环境（Python 3.11 兼容性好）----------
conda create -n cvcourse python=3.11 -y

# ---------- 3) 激活 / 退出环境 ----------
conda activate cvcourse
conda deactivate

# ---------- 4) 常用环境管理命令 ----------
conda env list                # 列出所有环境
conda list                    # 列出当前环境已装的包
conda remove -n cvcourse --all -y     # 删除整个环境
conda env export > environment.yml    # 导出环境清单（可复现）
conda env create -f environment.yml   # 用清单重建环境
```

### 1.1.3 用 pip 安装本课依赖

> ⚠️ **`opencv-python` vs `opencv-python-headless`——二者只能装一个！**
> - `opencv-python`：完整版，含 `cv2.imshow` 等 GUI 弹窗功能；
> - `opencv-python-headless`：**无 GUI** 版，体积更小，适合服务器 / Docker / CI / Notebook。
>
> 同时安装会互相覆盖 `cv2` 模块导致不可预知行为。若误装两者，先
> `pip uninstall opencv-python opencv-python-headless -y`，再单独重装一个。
>
> 本课程统一使用 **headless** 版（Notebook 里用 matplotlib 显示，不需要弹窗）。

在**已激活** `cvcourse` 环境后执行：

```bash
pip install numpy                    # 数组计算，一切的地基
pip install opencv-python-headless   # OpenCV（无 GUI 版，Notebook 推荐）
pip install pillow                   # PIL 的现代维护版
pip install matplotlib               # 绘图 / 图像显示
pip install scipy                    # 科学计算，ndimage 提供滤波与卷积
pip install jupyter                  # Notebook 运行环境
```

> 💡 课程目录 `实践课/` 下还提供了 `environment.yml`（conda 格式）与 `requirements.txt`（pip 格式），
> 可直接用 `conda env create -f environment.yml` 或 `pip install -r requirements.txt` 一键搭建环境。

In [ ]:
# ---------- 1.1.4 检查各依赖库是否安装成功 ----------
import platform
import sys

print("Python 版本 :", sys.version.split()[0])
print("解释器路径  :", sys.executable)
print("操作系统    :", platform.platform())

# 核心依赖（学生必须安装）
REQUIRED = [
    ("NumPy",      "numpy",      "numpy"),
    ("OpenCV",     "cv2",        "opencv-python-headless"),
    ("Pillow",     "PIL",        "pillow"),
    ("Matplotlib", "matplotlib", "matplotlib"),
    ("SciPy",      "scipy",      "scipy"),
]
# 选装（教师 / 自动化验证用，学生不装也不影响课程内容）
OPTIONAL = [
    ("nbformat",   "nbformat",   "nbformat"),
    ("ipykernel",  "ipykernel",  "ipykernel"),
]

print("\n核心依赖检查（✗ 表示未安装，按右侧命令补装）：")
missing = []
for display, module, pip_name in REQUIRED:
    try:
        mod = __import__(module)
        ver = getattr(mod, "__version__", "未知版本")
        print(f"  ✓ {display:<11s} {ver}")
    except ImportError:
        missing.append(pip_name)
        print(f"  ✗ {display:<11s} 未安装   ->  pip install {pip_name}")

print("\n选装依赖（教师/自动化验证用）：")
for display, module, pip_name in OPTIONAL:
    try:
        mod = __import__(module)
        ver = getattr(mod, "__version__", "未知版本")
        print(f"  ✓ {display:<11s} {ver}")
    except ImportError:
        print(f"  ○ {display:<11s} 未安装（不影响课程运行）")

if missing:
    print("\n请在终端执行：pip install " + " ".join(missing))
else:
    print("\n所有核心依赖均已就绪 🎉")

In [ ]:
# ---------- 1.1.5 查看 OpenCV 构建信息与当前绘图后端 ----------
import re

import cv2
import matplotlib

build = cv2.getBuildInformation()
m = re.search(r"GUI:\s*(\S+)", build)
gui_backend = m.group(1) if m else "NONE"
print("OpenCV 版本      :", cv2.__version__)
print("OpenCV GUI 支持  :", gui_backend,
      "（NONE 表示 headless 无 GUI 构建）")
print("Matplotlib 版本  :", matplotlib.__version__)
print("Matplotlib 后端  :", matplotlib.get_backend(),
      "（Agg = 只生成位图不弹窗，适合 Notebook/服务器）")


def in_notebook():
    """判断当前是否运行在 Jupyter Notebook 内核中。"""
    try:
        from IPython import get_ipython
        return get_ipython().__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


def cv2_gui_usable():
    """粗略判断 cv2.imshow 是否可用：需要 OpenCV 带 GUI 且不在 Notebook 中。"""
    return gui_backend.upper() != "NONE" and not in_notebook()


print("当前在 Notebook  :", in_notebook())
print("cv2.imshow 可用  :", cv2_gui_usable(),
      "→ 本课统一改用 matplotlib 显示" if not cv2_gui_usable() else "")

**运行结果说明**：

- 第一个单元会列出 7 个依赖的版本号，全部打 `✓` 即环境合格；
- 第二个单元里 `OpenCV GUI 支持` 若为 `NONE`，说明是 headless 构建；`Matplotlib 后端` 在 Notebook 中通常是 `module://matplotlib_inline.backend_inline`（内联显示），在纯脚本/服务器中是 `Agg`。

> 🔑 关键结论：**只要 `in_notebook()` 为 True，就永远不要用 `cv2.imshow`**，原因见下一节。

---

## 1.2 OpenCV：读写、显示、颜色空间与绘图

OpenCV（Open Source Computer Vision Library）是工业界使用最广的视觉库，Python 接口为 `cv2`。它有两条**必须记住**的约定：

1. **通道顺序是 BGR，不是 RGB**。这是历史遗留（早期 Windows 位图格式），用 `matplotlib` 显示前必须转换，否则红蓝互换；
2. **图像就是 NumPy 数组**，`img.shape == (高 H, 宽 W, 通道 C)`，`img.dtype` 通常是 `uint8`。因此所有 NumPy 技巧都能直接用在 OpenCV 图像上。

### 1.2.1 中文路径问题

`cv2.imread` / `cv2.imwrite` 在 **Windows** 上对含中文、空格等非 ASCII 字符的路径会**静默失败**（`imread` 返回 `None`，`imwrite` 返回 `False`），因为它内部用 ANSI 编码打开文件。macOS / Linux 一般正常。

**跨平台安全写法**：读用 `np.fromfile + cv2.imdecode`，写用 `cv2.imencode + ndarray.tofile`。下面把它封装成两个工具函数，全课复用。

In [ ]:
# ---------- 1.2.2 中文路径安全读写工具 ----------


def imread_unicode(path, flags=None):
    """跨平台读取图像，支持中文/空格路径。失败返回 None（不抛异常）。

    原理：np.fromfile 以二进制字节读入（不受路径编码影响），
    再交给 cv2.imdecode 在内存中解码。
    """
    if flags is None:
        flags = cv2.IMREAD_COLOR
    if not os.path.isfile(path):
        return None
    data = np.fromfile(path, dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def imwrite_unicode(path, img_bgr):
    """跨平台保存图像，支持中文路径。成功返回 True。

    原理：cv2.imencode 把图像编码成内存字节流，再用 tofile 写盘。
    注意：传入的必须是 BGR 顺序（OpenCV 约定）。
    """
    ext = os.path.splitext(path)[1] or ".png"
    ok, buf = cv2.imencode(ext, img_bgr)
    if not ok:
        return False
    buf.tofile(path)
    return True


# 把主演示图（RGB）转成 BGR 后，保存到一个"中文文件名"，验证工具函数
bgr_demo = cv2.cvtColor(RGB_IMG, cv2.COLOR_RGB2BGR)
cn_path = os.path.join(WORK_DIR, "演示图像_中文路径测试.png")
ok = imwrite_unicode(cn_path, bgr_demo)
print(f"保存到中文路径 {os.path.basename(cn_path)} : {'成功' if ok else '失败'}")

# 用两种方式读回来对比
via_plain = cv2.imread(cn_path)          # Windows 上这里会是 None
via_safe = imread_unicode(cn_path)
print("cv2.imread(中文路径)      :", "None（失败）" if via_plain is None else f"成功 {via_plain.shape}")
print("imread_unicode(中文路径)  :", "None（失败）" if via_safe is None else f"成功 {via_safe.shape}")
print("两种方式结果是否一致      :",
      (via_plain is None and via_safe is None)
      or (via_plain is not None and via_safe is not None
          and np.array_equal(via_plain, via_safe)))

**运行结果说明**：在 macOS / Linux 上两行都会显示「成功」且结果一致；在 **Windows** 上第一行会显示 `None（失败）`，第二行仍成功——这正是封装 `imread_unicode` 的价值。

### 1.2.3 显示图像：为什么不用 `cv2.imshow`？

`cv2.imshow` 会**新建一个操作系统窗口并阻塞**，必须配合 `cv2.waitKey` 使用，否则会一闪而过或卡死：

```python
# ⚠️ 以下代码在 Notebook 中【不要】运行，仅作说明
# cv2.imshow("demo", img)      # 弹出窗口
# cv2.waitKey(0)               # 等待按键；参数 0 表示无限等待
# cv2.destroyAllWindows()      # 关闭所有窗口
```

在 Notebook 里有三个致命问题：

1. **headless 构建根本没有 GUI**，调用直接抛 `cv2.error: The function is not implemented`；
2. 即使有 GUI，`waitKey(0)` 会**阻塞内核**，整个 Notebook 卡住，只能重启内核；
3. 弹窗无法内联进 Notebook，导出 PDF / HTML 时图像丢失。

**结论：Notebook 中一律用 `matplotlib.pyplot.imshow` 显示。** 本课程已封装好 `show_images()`，支持多图对比、中文标题、colorbar、保存。

In [ ]:
# ---------- 1.2.4 显示图像 + BGR/RGB 顺序陷阱 ----------
# cv2 读回来的是 BGR；直接丢给 matplotlib 会红蓝互换
wrong = via_safe                                   # 仍是 BGR
right = cv2.cvtColor(via_safe, cv2.COLOR_BGR2RGB)  # 转成 RGB 才正确

show_images([wrong, right],
            titles=["错误：BGR 直接显示（红蓝互换）", "正确：先 cvtColor 转 RGB"],
            cols=2, suptitle="OpenCV 的 BGR 顺序陷阱")

**运行结果说明**：左图会呈现诡异的**偏色**（因为 R/B 通道被交换），右图才是正常色彩。记住口诀：**「OpenCV 进 BGR，Matplotlib 出 RGB」**。

In [ ]:
# ---------- 1.2.5 颜色空间转换 cvtColor ----------
bgr = via_safe
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

print("BGR  :", bgr.shape, bgr.dtype)
print("RGB  :", rgb.shape)
print("GRAY :", gray.shape, "（通道数从 3 变 1）")
print("HSV  :", hsv.shape, " H∈[0,179] S∈[0,255] V∈[0,255]")

show_images([rgb, gray, hsv[..., 0], hsv[..., 1], hsv[..., 2]],
            titles=["RGB", "灰度 GRAY", "HSV-H 色调", "HSV-S 饱和度", "HSV-V 明度"],
            cols=5, suptitle="常用颜色空间及其通道")

**运行结果说明**：

- `GRAY` 的 `shape` 是 `(640, 640)`（与读入的彩色图同尺寸），**通道维度消失**（不是 `(640,640,1)`）；
- HSV 中 **H（色调）** 在 OpenCV 里取值 `0~179`（真实角度的一半，为了塞进 uint8），不同色相区域的 H 值差异明显；**S（饱和度）** 反映颜色纯度，接近无彩色的区域 S≈0；**V（明度）** 接近灰度图。

> 💡 HSV 是颜色分割的利器：要提取「橙色夕阳」，只需对 H 通道做阈值，比在 RGB 空间做三通道阈值稳健得多。

In [ ]:
# ---------- 1.2.6 图像缩放 resize 与插值方法 ----------
sizes = [(120, 120), (240, 240)]
methods = [
    ("INTER_NEAREST 最近邻", cv2.INTER_NEAREST),
    ("INTER_LINEAR 双线性", cv2.INTER_LINEAR),
    ("INTER_CUBIC 三次插值", cv2.INTER_CUBIC),
    ("INTER_AREA 区域(缩图推荐)", cv2.INTER_AREA),
    ("INTER_LANCZOS4", cv2.INTER_LANCZOS4),
]
imgs, titles = [], []
for name, flag in methods:
    small = cv2.resize(rgb, sizes[0], interpolation=cv2.INTER_NEAREST)
    back = cv2.resize(small, (rgb.shape[1], rgb.shape[0]), interpolation=flag)  # 放大回原尺寸看锯齿
    imgs.append(back)
    titles.append(name)
show_images(imgs, titles=titles, cols=5,
            suptitle=f"先缩到 120×120 再放大回 {rgb.shape[1]}×{rgb.shape[0]}：不同插值的差异")

**运行结果说明**：`INTER_NEAREST` 放大后边缘呈明显**马赛克锯齿**；`INTER_LINEAR/CUBIC/LANCZOS4` 边缘平滑但略有模糊。经验法则：

- **缩小**图像用 `INTER_AREA`（抗混叠最好）；
- **放大**图像用 `INTER_CUBIC` 或 `INTER_LANCZOS4`；
- 需要**保持硬边缘/标签图**（如分割 mask）用 `INTER_NEAREST`，避免插值产生不存在的类别值。

In [ ]:
# ---------- 1.2.7 基本绘图 ----------
canvas = rgb.copy()
H, W = canvas.shape[:2]

cv2.line(canvas, (20, 20), (W - 20, 120), (255, 0, 0), thickness=4)            # 直线(红)
cv2.rectangle(canvas, (40, 160), (200, 300), (0, 255, 0), thickness=3)         # 矩形(绿)
cv2.circle(canvas, (330, 230), 70, (0, 0, 255), thickness=-1)                  # 实心圆(蓝, -1=填充)
cv2.ellipse(canvas, (240, 400), (120, 40), 20, 0, 360, (255, 255, 0), 3)       # 椭圆(黄)
pts = np.array([[60, 430], [140, 360], [220, 430]], np.int32).reshape(-1, 1, 2)
cv2.polylines(canvas, [pts], isClosed=True, color=(255, 0, 255), thickness=3)  # 多边形(品红)
cv2.putText(canvas, "OpenCV Draw", (60, 90), cv2.FONT_HERSHEY_SIMPLEX,
            1.2, (255, 255, 255), 3, cv2.LINE_AA)                              # 文字(白)

show_images([rgb, canvas], titles=["原图", "绘图结果"], cols=2,
            suptitle="cv2 基本绘图：line / rectangle / circle / ellipse / polylines / putText")

**运行结果说明**：注意 `cv2` 的坐标约定是 **`(x, y)`，即 (列, 行)**，与 NumPy 索引 `img[y, x]` 相反。**本画布 `canvas` 已是 RGB 数组**（由 `rgb.copy()` 得来），所以颜色元组按 **(R, G, B)** 解读——`(255,0,0)` 画出**红色**直线，`(0,0,255)` 画出**蓝色**实心圆。若你直接在 OpenCV 读入的 BGR 数组上绘图，则元组含义反转（`(255,0,0)` 变蓝）。`thickness=-1` 表示填充。`putText` 不支持中文，中文标注请用 matplotlib。

In [ ]:
# ---------- 1.2.8 保存图像 ----------
out_rgb = os.path.join(WORK_DIR, "opencv_saved_rgb.png")
out_gray = os.path.join(WORK_DIR, "opencv_saved_gray.png")
# imwrite 期望 BGR：把 RGB 转回去再存，文件才是正常颜色
print("保存彩色 :", imwrite_unicode(out_rgb, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)))
print("保存灰度 :", imwrite_unicode(out_gray, gray))
for p in (out_rgb, out_gray):
    print(f"  {os.path.basename(p)}  存在={os.path.isfile(p)}  "
          f"大小={os.path.getsize(p) // 1024} KB")

**运行结果说明**：两个文件被写入 `实践课/_output/` 目录。灰度图保存后只有 1 个通道，文件体积明显更小。

> 📌 **§1.2 小结**
> | 操作 | 函数 | 易错点 |
> | :--- | :--- | :--- |
> | 读图 | `cv2.imread` / `imread_unicode` | 中文路径用 `imdecode+fromfile` |
> | 显示 | `matplotlib.imshow` | 先 `cvtColor(BGR2RGB)` |
> | 存图 | `cv2.imwrite` / `imwrite_unicode` | 输入需为 BGR |
> | 颜色空间 | `cv2.cvtColor` | 灰度后通道维消失 |
> | 缩放 | `cv2.resize` | 缩图用 AREA，放大用 CUBIC |
> | 绘图 | `line/rectangle/circle/...` | 坐标是 `(x,y)`，颜色是 BGR |

---

## 1.3 PIL (Pillow)：面向「图像文件」的轻量工具

Pillow 是经典 PIL（Python Imaging Library）的现代维护版。它与 OpenCV 的**定位不同**：

| 维度 | OpenCV (`cv2`) | Pillow (`PIL`) |
| :--- | :--- | :--- |
| 核心抽象 | **NumPy 数组**（像素矩阵） | **`Image` 对象**（封装文件+像素+元数据） |
| 强项 | 实时视觉算法、视频、特征、标定 | 读写各种格式、EXIF、简单编辑、缩略图 |
| 通道顺序 | BGR | **RGB** |
| 尺寸表示 | `img.shape = (H, W, C)` | `img.size = (W, H)` ⚠️ 相反 |
| 中文路径 | Windows 上需 `imdecode` 绕开 | **原生支持** |
| 典型场景 | 检测/跟踪/标定/实时处理 | 格式转换/加水印/缩略图/网页图处理 |

> 💡 实践建议：**文件 I/O 和简单编辑用 PIL，数值算法用 OpenCV/NumPy**，二者通过 ndarray 无缝互转。

In [ ]:
# ---------- 1.3.1 打开图像与基本属性 ----------
from PIL import Image, ImageEnhance, ImageFilter

pil_img = Image.open(cn_path)          # PIL 原生支持中文路径，无需任何绕开手段
print("类型      :", type(pil_img))
print("size(W,H) :", pil_img.size, "  ⚠️ 注意是 (宽, 高)")
print("mode      :", pil_img.mode, "   (RGB / L / RGBA / P / 1 ...)")
print("format    :", pil_img.format)
print("info 键   :", list(pil_img.info.keys()))

arr = np.asarray(pil_img)
print("转 ndarray 后 shape(H,W,C) :", arr.shape, "  ⚠️ 与 size 顺序相反")
print("同一张图：PIL.size =", pil_img.size, " vs ndarray.shape =", arr.shape)

show_images([np.asarray(pil_img)], titles=["PIL 打开的中文路径图像"], cols=1)

In [ ]:
# ---------- 1.3.2 模式转换 convert ----------
modes = [("RGB 原图", "RGB"), ("L 灰度", "L"), ("1 二值", "1"),
         ("P 调色板", "P"), ("HSV", "HSV"), ("RGBA 带透明", "RGBA")]
imgs, titles = [], []
for title, mode in modes:
    conv = pil_img.convert(mode)
    # 统一转回 RGB 才能并排显示（'1'/'L'/'P' 显示时 matplotlib 会自动处理灰度）
    imgs.append(np.asarray(conv.convert("RGB") if mode in ("P", "RGBA") else conv))
    titles.append(f"{title}\nmode={conv.mode}")
show_images(imgs, titles=titles, cols=6, suptitle="Image.convert 模式转换对比")

print("本次演示涉及的模式：", [m for _, m in modes])
gray_pil = pil_img.convert("L")
print("转 'L' 后 mode =", gray_pil.mode, ", 数组 shape =", np.asarray(gray_pil).shape)

**运行结果说明**：

- `convert("L")` 用加权公式 `L = 0.299R + 0.587G + 0.114B` 得到灰度，与 §0 的 `load_demo_gray` 一致；
- `convert("1")` 是**纯黑白二值**（每像素 1 bit），中间调全部丢失，画面只剩黑白块；
- `convert("P")` 量化成调色板色，颜色数骤减，会出现色带；
- `convert("RGBA")` 增加 Alpha 透明通道，`shape` 变为 `(H, W, 4)`。

In [ ]:
# ---------- 1.3.3 缩放 resize / thumbnail ----------
orig_w, orig_h = pil_img.size
resized = pil_img.resize((orig_w // 4, orig_h // 4), Image.LANCZOS)
print("resize 后 size :", resized.size)

thumb = pil_img.copy()
thumb.thumbnail((128, 128), Image.LANCZOS)   # 保持宽高比，且【原地修改】
print("thumbnail 后 size :", thumb.size, "（等比缩放，最长边 ≤128）")

resample_methods = [("NEAREST", Image.NEAREST), ("BILINEAR", Image.BILINEAR),
                    ("BICUBIC", Image.BICUBIC), ("LANCZOS", Image.LANCZOS)]
imgs, titles = [], []
for name, method in resample_methods:
    small = pil_img.resize((96, 96), method)
    imgs.append(np.asarray(small.resize((384, 384), Image.NEAREST)))  # 放大回来看差异
    titles.append(name)
show_images(imgs, titles=titles, cols=4,
            suptitle="缩到 96×96 再最近邻放大：PIL 重采样方法对比")

**运行结果说明**：`thumbnail` 与 `resize` 的关键区别是 **`thumbnail` 保持宽高比且原地修改、不返回新图**；`resize` 返回新图且可任意拉伸（可能变形）。重采样方法中 `LANCZOS` 缩图质量最好，`NEAREST` 最快但锯齿最重。

In [ ]:
# ---------- 1.3.4 旋转 / 翻转 / 裁剪 ----------
rot_keep = pil_img.rotate(30)                    # 不 expand：画布不变，四角补黑
rot_expand = pil_img.rotate(30, expand=True)     # expand：画布扩大以容纳整图
flip = pil_img.transpose(Image.FLIP_LEFT_RIGHT)  # 水平镜像
crop_box = (orig_w // 4, orig_h // 4, orig_w * 3 // 4, orig_h * 3 // 4)  # (left, top, right, bottom)
cropped = pil_img.crop(crop_box)

print("rotate(30)          size =", rot_keep.size)
print("rotate(30,expand)   size =", rot_expand.size, "（画布变大）")
print("crop(box)           size =", cropped.size, " box =", crop_box)

show_images([np.asarray(rot_keep), np.asarray(rot_expand),
             np.asarray(flip), np.asarray(cropped)],
            titles=["rotate(30) 补黑角", "rotate(30, expand=True)",
                    "FLIP_LEFT_RIGHT 镜像", f"crop{crop_box}"],
            cols=4, suptitle="几何变换")

**运行结果说明**：`rotate` 默认 `expand=False`，旋转后**画布尺寸不变**，超出部分被裁掉、空出部分填黑；`expand=True` 会扩大画布完整保留图像。`crop` 的 box 是 **`(left, top, right, bottom)`**，即 `(x0, y0, x1, y1)`，与 NumPy 切片 `img[y0:y1, x0:x1]` 的轴顺序相反。

In [ ]:
# ---------- 1.3.5 内置滤镜 ImageFilter ----------
filters = [
    ("原图", None),
    ("BLUR", ImageFilter.BLUR),
    ("GaussianBlur(2)", ImageFilter.GaussianBlur(2)),
    ("SHARPEN", ImageFilter.SHARPEN),
    ("DETAIL", ImageFilter.DETAIL),
    ("SMOOTH", ImageFilter.SMOOTH),
    ("EDGE_ENHANCE_MORE", ImageFilter.EDGE_ENHANCE_MORE),
    ("FIND_EDGES", ImageFilter.FIND_EDGES),
    ("EMBOSS", ImageFilter.EMBOSS),
    ("CONTOUR", ImageFilter.CONTOUR),
    ("MedianFilter(5)", ImageFilter.MedianFilter(5)),
    ("MaxFilter(5)", ImageFilter.MaxFilter(5)),
]
imgs, titles = [], []
for name, f in filters:
    out = pil_img if f is None else pil_img.filter(f)
    imgs.append(np.asarray(out.convert("RGB")))
    titles.append(name)
show_images(imgs, titles=titles, cols=4, suptitle="PIL ImageFilter 滤镜全家福")

**运行结果说明**：

- `BLUR` / `GaussianBlur` / `SMOOTH` 是**低通**，抹平细节与噪声；`GaussianBlur(radius)` 的 radius 越大越糊；
- `SHARPEN` / `DETAIL` / `EDGE_ENHANCE_MORE` 是**高通增强**，突出边缘；
- `FIND_EDGES` / `CONTOUR` / `EMBOSS` 输出**边缘/浮雕**图，背景接近黑或灰；
- `MedianFilter(5)` 是**中值滤波**，对椒盐噪声特别有效（§1.4.3 会深入）；
- `MaxFilter(5)` 是**膨胀**，亮区扩张。

> ⚠️ PIL 的滤镜是**固定核**，不能自定义卷积核；要自定义核请用 §1.4 的 `scipy.ndimage` 或 §1.5 的 NumPy。

In [ ]:
# ---------- 1.3.5b 图像增强 ImageEnhance（亮度/对比度/色彩/锐度）----------
enhancers = [
    ("Brightness 0.5", ImageEnhance.Brightness, 0.5),
    ("Brightness 1.5", ImageEnhance.Brightness, 1.5),
    ("Contrast 0.5", ImageEnhance.Contrast, 0.5),
    ("Contrast 1.8", ImageEnhance.Contrast, 1.8),
    ("Color 0 (去色)", ImageEnhance.Color, 0.0),
    ("Color 2.0", ImageEnhance.Color, 2.0),
    ("Sharpness 3.0", ImageEnhance.Sharpness, 3.0),
]
imgs, titles = [np.asarray(pil_img)], ["原图 factor=1.0"]
for name, cls, factor in enhancers:
    imgs.append(np.asarray(cls(pil_img).enhance(factor).convert("RGB")))
    titles.append(f"{name}")
show_images(imgs, titles=titles, cols=4, suptitle="ImageEnhance：factor<1 减弱，>1 增强")

**运行结果说明**：所有 Enhancer 都遵循统一接口 `cls(img).enhance(factor)`，`factor=1.0` 为原图，`0` 为完全减弱（如 `Color(0)` 得到灰度效果），`>1` 为增强。

In [ ]:
# ---------- 1.3.6 与 NumPy / OpenCV 互转 ----------
# PIL -> NumPy
arr_from_pil = np.asarray(pil_img)          # 零拷贝视图（RGB, uint8）
arr_copy = np.array(pil_img)                # 强制拷贝，安全可改
print("PIL->ndarray :", arr_from_pil.shape, arr_from_pil.dtype)

# NumPy -> PIL（dtype 必须是 uint8，mode 由 shape 推断）
back_to_pil = Image.fromarray(arr_copy)
print("ndarray->PIL :", back_to_pil.size, back_to_pil.mode)
print("往返是否无损 :", np.array_equal(np.asarray(back_to_pil), arr_copy))

# 视图 vs 拷贝：np.asarray 得到的视图与 PIL 共享内存，np.array 的拷贝则独立
view = np.asarray(pil_img)
copy = np.array(pil_img)
print("asarray 是视图(共享内存) :", view.base is not None or view is np.asarray(pil_img))
copy[0, 0] = [255, 0, 0]
print("改 copy 不影响 PIL 原图  :", tuple(pil_img.getpixel((0, 0))) != (255, 0, 0))

# PIL(RGB) <-> OpenCV(BGR)
pil_rgb = np.asarray(pil_img)                       # RGB
cv2_bgr = cv2.cvtColor(pil_rgb, cv2.COLOR_RGB2BGR)  # 转成 OpenCV 约定
pil_back = cv2.cvtColor(cv2_bgr, cv2.COLOR_BGR2RGB)
print("PIL<->OpenCV 往返无损 :", np.array_equal(pil_rgb, pil_back))
# 也可用切片快速互换通道（等价于 cvtColor，但更快）
print("切片互换等价 cvtColor :", np.array_equal(pil_rgb[..., ::-1], cv2_bgr))

**运行结果说明**：

- `np.asarray(pil_img)` 通常是**只读视图**（不复制内存），`np.array(pil_img)` 才是可安全修改的**拷贝**；
- `Image.fromarray` 要求 `uint8`，传入 float 会报错或得到意外 mode，务必先 `.astype(np.uint8)`；
- PIL 是 RGB、OpenCV 是 BGR，互转用 `cv2.cvtColor` 或切片 `[..., ::-1]`（二者等价）。

> 📌 **§1.3 小结（PIL vs OpenCV 选择指南）**
> | 需求 | 推荐 |
> | :--- | :--- |
> | 读各种格式 / EXIF / 中文路径 | PIL |
> | 缩略图、旋转、加水印、格式转换 | PIL |
> | 固定滤镜（模糊/锐化/边缘） | PIL |
> | 自定义卷积核、滤波、形态学 | OpenCV / SciPy |
> | 实时视频、特征点、标定 | OpenCV |
> | 像素级数学运算 | NumPy（二者皆可转） |

---

## 1.4 SciPy：`scipy.ndimage` 的滤波与卷积

`scipy.ndimage` 是 SciPy 的**多维图像处理**子模块。它的特点是：

- 纯 NumPy 数组接口，**没有 BGR/RGB 的包袱**（它不关心颜色语义，只把最后一维当普通轴或逐通道处理）；
- 提供**可自定义卷积核**的 `convolve` / `correlate`，以及大量现成滤波（`uniform_filter`、`gaussian_filter`、`median_filter`…）；
- 边界处理策略统一通过 `mode` 参数控制，非常适合讲清楚「卷积到底在边界做了什么」。

本课统一把图像转成 `float64` 再滤波（避免 uint8 截断），显示/保存前再 `clip` 回 `[0,255]` 并转 `uint8`。

In [ ]:
# ---------- 1.4.0 准备：灰度浮点图像 ----------
from scipy import ndimage

G = GRAY_IMG.astype(np.float64)          # (H, W) float64，与 GRAY_IMG 同尺寸（当前 640×640）
print("滤波用灰度图 G :", G.shape, G.dtype, " 值域", G.min(), "~", G.max())


def to_uint8(arr):
    """把浮点滤波结果安全地转回 uint8 以便显示/保存。"""
    return np.clip(arr, 0, 255).astype(np.uint8)

### 1.4.1 图像模糊（低通滤波）

模糊的本质是**低通滤波**：抑制高频（细节、噪声），保留低频（大尺度结构）。两种最常用的低通：

- **均值滤波 `uniform_filter`**：邻域内取算术平均，核为全 1 矩阵。快，但会产生「方块感」振铃；
- **高斯滤波 `gaussian_filter`**：邻域内按高斯权重加权平均，越靠近中心权重越大。平滑更自然，是去高斯噪声的首选。

In [ ]:
# ---------- 1.4.1a 均值滤波 uniform_filter：size 越大越糊 ----------
sizes = [1, 3, 5, 9, 15, 25]
imgs, titles = [], []
for s in sizes:
    out = ndimage.uniform_filter(G, size=s) if s > 1 else G
    imgs.append(to_uint8(out))
    titles.append(f"uniform size={s}")
show_images(imgs, titles=titles, cols=6, suptitle="均值滤波：邻域越大，细节丢失越多")

In [ ]:
# ---------- 1.4.1b 高斯滤波 gaussian_filter：sigma 控制平滑强度 ----------
sigmas = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
imgs, titles = [], []
for sg in sigmas:
    out = ndimage.gaussian_filter(G, sigma=sg)
    imgs.append(to_uint8(out))
    titles.append(f"gaussian σ={sg}")
show_images(imgs, titles=titles, cols=6, suptitle="高斯滤波：σ 越大越糊（σ≈核半径的 1/3）")

**运行结果说明**：

- `uniform_filter(size=s)` 的 `s` 是**正方形邻域边长**；`s=1` 等于不滤波；
- `gaussian_filter(sigma=σ)` 的 `σ` 决定高斯核的「胖瘦」，SciPy 会自动按 `truncate=4.0`（即核半径 ≈ 4σ）截断核；
- 相同「视觉糊度」下，高斯比均值**更不容易产生振铃和方块伪影**。

In [ ]:
# ---------- 1.4.1c 均值 vs 高斯 + 边界模式 mode ----------
uni = ndimage.uniform_filter(G, size=9)
gau = ndimage.gaussian_filter(G, sigma=3)

modes = ["reflect", "constant", "nearest", "wrap", "mirror"]
imgs = [to_uint8(G)] + [to_uint8(ndimage.gaussian_filter(G, sigma=6, mode=m)) for m in modes]
titles = ["原图"] + [f"mode={m}" for m in modes]
show_images(imgs, titles=titles, cols=6,
            suptitle="高斯 σ=6 在不同边界模式下的边缘表现（观察四周亮暗）")

print("均值 vs 高斯（size=9 / σ=3）的差异 MSE :", round(mse(uni, gau), 2))
print("mode 说明：reflect=镜像反射(默认) | constant=补 cval | nearest=复制边缘 | wrap=循环 | mirror=不含边界的镜像")

**运行结果说明**：边界模式差异在**图像四周**最明显：

- `constant`（默认补 0，即黑）会让边缘出现一圈**暗边**；
- `reflect` / `mirror` 用镜像像素填充，边缘过渡自然（默认 `reflect`）；
- `nearest` 复制最外圈像素；`wrap` 把图像当周期信号首尾相接，通常不用于自然图像。

> 💡 做定量实验（如 PSNR）时，边界填充会引入误差。若只关心内部区域，可裁掉边界若干像素再比较。

### 1.4.2 图像模板算子：卷积 `convolve` 与相关 `correlate`

**卷积**与**相关**是线性滤波的数学基础。二者唯一的区别：卷积会先把核翻转 180°再做相关。
对于对称核（均值、高斯）二者结果相同；对于非对称核则不同——理解这一点非常重要。

In [ ]:
# ---------- 1.4.2a 定义常用卷积核 ----------
K_MEAN3 = np.ones((3, 3), np.float64) / 9.0
K_MEAN31 = np.ones((31, 31), np.float64) / 961.0
K_GAUSSIAN = np.array([[1, 2, 1],
                       [2, 4, 2],
                       [1, 2, 1]], np.float64) / 16.0   # 近似高斯
# 非对称核（用于演示卷积 vs 相关的区别）
K_ASYM = np.array([[1, 1, 1],
                   [0, 0, 0],
                   [-1, -1, -1]], np.float64)             # 只取上一行加权

for name, K in [("均值3×3", K_MEAN3), ("均值31×31", K_MEAN31),
                ("高斯3×3", K_GAUSSIAN), ("非对称", K_ASYM)]:
    print(f"{name:>8s}: sum={K.sum():.4f}  shape={K.shape}")


**运行结果说明**：**核元素之和**决定了滤波的「直流增益」：

- 和为 **1**（均值、高斯）→ 保持整体亮度不变；
- 和为 **0** 的核 → 抑制恒定区域、只响应变化，是高通/微分算子（后续课程讲授）。
- 上面的非对称核 `K_ASYM` 和为 6（非 1 也非 0），用于下一步演示卷积与相关的差别。

In [ ]:
# ---------- 1.4.2b 卷积 vs 相关：用非对称核看差别 ----------
conv_a = ndimage.convolve(G, K_ASYM, mode="reflect")
corr_a = ndimage.correlate(G, K_ASYM, mode="reflect")
# 理论：convolve(f, w) == correlate(f, w 旋转180°)
corr_flip = ndimage.correlate(G, K_ASYM[::-1, ::-1], mode="reflect")

print("convolve 与 correlate 是否相同      :", np.allclose(conv_a, corr_a))
print("convolve(f,w) == correlate(f,flip(w)):", np.allclose(conv_a, corr_flip))

# 对称核则两者一致
conv_s = ndimage.convolve(G, K_GAUSSIAN, mode="reflect")
corr_s = ndimage.correlate(G, K_GAUSSIAN, mode="reflect")
print("对称核 Gaussian 下 convolve==correlate:", np.allclose(conv_s, corr_s))

show_images([conv_a, corr_a],
            titles=["|convolve(f, K_ASYM)|", "|correlate(f, K_ASYM)|"], cols=2,
            suptitle="非对称核下卷积与相关结果方向相反")

In [ ]:
# ---------- 1.4.2c 边界处理 mode / cval 对卷积的影响 ----------
imgs = [to_uint8(ndimage.correlate(G, K_MEAN31, mode=m, cval=0.0))
        for m in ["reflect", "constant", "nearest", "wrap"]]
show_images(imgs, titles=[f"均值5×5, mode={m}" for m in ["reflect", "constant", "nearest", "wrap"]],
            cols=4, suptitle="边界模式如何影响滤波结果（观察四周一圈）")


**运行结果说明**：`mode="constant"` 时图像外被当作 0（黑），于是**边界一圈会变暗**（被零值拉低）；`reflect`/`nearest` 则用图像自身边缘信息填充，结果更自然。这就是为什么默认推荐 `reflect`。

### 1.4.3 实践：图像去噪

去噪是滤波最经典的应用。先认识两种噪声模型：

| 噪声 | 模型 | 视觉特征 | 克星 |
| :--- | :--- | :--- | :--- |
| **高斯噪声** | $g = f + n,\ n\sim\mathcal N(0,\sigma^2)$ | 整幅图蒙一层细砂 | 高斯/均值低通 |
| **椒盐噪声** | 随机像素置 0 或 255 | 黑白杂点（坏点） | **中值滤波** |

关键洞察：**中值是非线性滤波**，它取邻域排序后的中间值，能把「离群的极亮/极暗点」直接剔除，同时**保住边缘**；而均值/高斯是线性加权，会把椒盐点「抹开」成一片灰斑。

In [ ]:
# ---------- 1.4.3a 生成两种噪声 ----------
clean = to_uint8(G)
gauss_noisy = add_gaussian_noise(clean, sigma=25, seed=7)
sp_noisy = add_salt_pepper_noise(clean, amount=0.05, seed=7)

show_images([clean, gauss_noisy, sp_noisy],
            titles=["干净原图", f"高斯噪声 σ=25\nPSNR={psnr(clean, gauss_noisy):.1f}dB",
                    f"椒盐噪声 5%\nPSNR={psnr(clean, sp_noisy):.1f}dB"],
            cols=3, suptitle="两种噪声模型")

In [ ]:
# ---------- 1.4.3b 高斯滤波去高斯噪声：σ 的选择 ----------
rows = []
imgs, titles = [gauss_noisy], ["含噪原图"]
for sg in [0.5, 1.0, 2.0, 3.0, 5.0]:
    den = to_uint8(ndimage.gaussian_filter(gauss_noisy.astype(np.float64), sigma=sg))
    p = psnr(clean, den)
    rows.append((f"gaussian σ={sg}", p))
    imgs.append(den)
    titles.append(f"σ={sg}\nPSNR={p:.1f}dB")
show_images(imgs, titles=titles, cols=6, suptitle="高斯滤波去高斯噪声：σ 太小去不净、太大过糊")

print("高斯噪声 -> 高斯滤波 PSNR：")
for name, p in rows:
    print(f"  {name:<16s} {p:6.2f} dB")

In [ ]:
# ---------- 1.4.3c 中值滤波去椒盐噪声：size 的选择 ----------
rows_sp = []
imgs, titles = [sp_noisy], ["含噪原图"]
for s in [3, 5, 7, 9]:
    den = ndimage.median_filter(sp_noisy, size=s)
    p = psnr(clean, den)
    rows_sp.append((f"median size={s}", p))
    imgs.append(den)
    titles.append(f"size={s}\nPSNR={p:.1f}dB")
show_images(imgs, titles=titles, cols=5, suptitle="中值滤波去椒盐噪声：size≥3 即可几乎完全去除")

print("椒盐噪声 -> 中值滤波 PSNR：")
for name, p in rows_sp:
    print(f"  {name:<16s} {p:6.2f} dB")

In [ ]:
# ---------- 1.4.3d 交叉实验：选错滤波器会怎样 ----------
gf = to_uint8(ndimage.gaussian_filter(gauss_noisy.astype(np.float64), sigma=2))
mf_g = ndimage.median_filter(gauss_noisy, size=3)          # 中值 去 高斯噪声
gf_sp = to_uint8(ndimage.gaussian_filter(sp_noisy.astype(np.float64), sigma=2))  # 高斯 去 椒盐
mf_sp = ndimage.median_filter(sp_noisy, size=3)            # 中值 去 椒盐

print("=== 交叉实验 PSNR（越高越好）===")
print(f"高斯噪声 + 高斯滤波 : {psnr(clean, gf):6.2f} dB   ← 对症")
print(f"高斯噪声 + 中值滤波 : {psnr(clean, mf_g):6.2f} dB")
print(f"椒盐噪声 + 高斯滤波 : {psnr(clean, gf_sp):6.2f} dB   ← 不对症，噪点被抹成灰斑")
print(f"椒盐噪声 + 中值滤波 : {psnr(clean, mf_sp):6.2f} dB   ← 对症")

show_images([gf, mf_g, gf_sp, mf_sp],
            titles=["高斯噪+高斯滤", "高斯噪+中值滤", "椒盐噪+高斯滤(灰斑!)", "椒盐噪+中值滤"],
            cols=4, suptitle="滤波器与噪声类型必须匹配")

In [ ]:
# ---------- 1.4.3e 定量汇总：PSNR 柱状图 ----------
import matplotlib.pyplot as plt

labels = ["高斯噪\n+高斯滤", "高斯噪\n+中值滤", "椒盐噪\n+高斯滤", "椒盐噪\n+中值滤"]
values = [psnr(clean, gf), psnr(clean, mf_g), psnr(clean, gf_sp), psnr(clean, mf_sp)]
colors = ["#2E9BD6", "#9aa7b1", "#d98880", "#27ae60"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.6)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}", ha="center", fontsize=10)
base = psnr(clean, gauss_noisy)
ax.axhline(base, ls="--", c="gray", lw=1)
ax.text(3.4, base, f" 含噪基线 {base:.1f}dB", va="center", fontsize=9, color="gray")
ax.set_ylabel("PSNR (dB)")
ax.set_title("去噪效果定量对比：滤波器必须与噪声类型匹配")
ax.set_ylim(0, max(values) * 1.2)
fig.tight_layout()
plt.show()

**运行结果说明与结论**：

1. **高斯噪声 → 高斯/均值低通**有效，但 σ 需与噪声强度匹配，过大会牺牲细节；
2. **椒盐噪声 → 中值滤波**几乎完美（`size=3` 就能把 PSNR 拉得很高），且**保边**；
3. **用高斯滤椒盐**是最典型的错误：黑白点被平均成灰斑，PSNR 提升有限且画面发脏；
4. PSNR 是客观指标，但**最终要看图**——PSNR 相近时人眼感受可能差别很大。

> 📌 **§1.4 小结**
> | 函数 | 作用 | 关键参数 |
> | :--- | :--- | :--- |
> | `uniform_filter` | 均值低通 | `size` |
> | `gaussian_filter` | 高斯低通 | `sigma`, `mode` |
> | `median_filter` | 中值（非线性，去椒盐） | `size` |
> | `convolve` | 卷积（核先翻转 180°） | `weights`, `mode`, `cval` |
> | `correlate` | 相关（核不翻转） | `weights`, `mode`, `cval` |

---

## 1.5 NumPy：把图像当作数组来「手搓」算法

前面三节我们「调用」库函数；本节我们**亲手实现**核心算法。理由有三：

1. 面试与科研中常被要求手写插值、卷积、锐化；
2. 只有手写一遍，才真正理解库函数每个参数的含义；
3. NumPy 的**向量化**思维是高效图像处理的基础——避免 Python 双层 for 循环。

> 🔑 核心思想：**图像 = ndarray**。一切点运算（灰度变换）是逐元素函数，一切邻域运算（卷积/滤波）是滑动窗口加权，一切几何运算（缩放）是坐标映射 + 插值。

### 1.5.1 灰度变换（点运算）

灰度变换对每个像素独立地做映射 $s = T(r)$，不涉及邻域。四类经典变换：

| 变换 | 公式 | 作用 |
| :--- | :--- | :--- |
| 反转 | $s = L-1-r$ | 负片，增强暗区细节 |
| 线性 | $s = a r + b$ | 对比度拉伸（$a>1$）/ 压缩（$a<1$） |
| 分段线性 | 折线 | 只拉伸感兴趣的灰度区间 |
| 伽马 | $s = c r^{\gamma}$ | 校正显示非线性；$\gamma<1$ 提亮 |

In [ ]:
# ---------- 1.5.1a 反转与线性变换 ----------
r = G                                        # float64 灰度
L = 256

inverted = L - 1 - r                         # 反转
lin_stretch = np.clip(1.6 * r - 60, 0, 255)  # a=1.6,b=-60：对比度拉伸
lin_compress = np.clip(0.5 * r + 60, 0, 255) # a=0.5,b=+60：对比度压缩、整体提亮

show_images([to_uint8(r), to_uint8(inverted), to_uint8(lin_stretch), to_uint8(lin_compress)],
            titles=["原图", "反转 s=255-r", "线性 a=1.6,b=-60\n(对比度拉伸)", "线性 a=0.5,b=+60\n(压缩提亮)"],
            cols=4, suptitle="反转与线性灰度变换")

print("线性变换 s = a*r + b：a 控制对比度斜率，b 控制亮度平移")
print("注意 np.clip 必不可少：a*r+b 会越出 [0,255]，uint8 会回绕(wrap-around)产生严重伪影")

### 1.5.2 图像缩放：手写插值

缩放 = **坐标反向映射 + 插值**。对输出图每个像素 $(x', y')$，反算它在原图的坐标：
$$ x = \frac{W}{W'} \left(x' + \tfrac12\right) - \tfrac12,\qquad y = \frac{H}{H'} \left(y' + \tfrac12\right) - \tfrac12 $$
（$+0.5$ 是为了让像素中心对齐，避免半像素偏移。）

- **最近邻**：取离 $(x,y)$ 最近的整数像素。快，但有马赛克；
- **双线性**：用 $(x,y)$ 周围 2×2 四个像素按距离加权平均。平滑，但略糊。

> ⚠️ **关于 +0.5 公式的适用范围**：上面的中心对齐公式 $x = \frac{W}{W'}(x'+0.5)-0.5$ 主要用于双线性等**连续插值**方法。
> 对于**最近邻**，OpenCV `INTER_NEAREST` 实际使用的是简化公式 $x = \lfloor x' \cdot W / W' \rfloor$（不加不减 0.5），
> 本课程的手写实现与此一致。如果你按 +0.5 公式实现最近邻，结果会和 OpenCV 有 1 像素的偏差——这不是 bug，而是设计差异。

In [ ]:
# ---------- 1.5.2a 最近邻插值（纯 NumPy 向量化）----------
def resize_nearest(img, new_h, new_w):
    """最近邻插值缩放。返回与输入同 dtype 的数组（灰度或彩色均可）。"""
    h, w = img.shape[:2]
    ys = np.clip((np.arange(new_h) * h / new_h).astype(int), 0, h - 1)
    xs = np.clip((np.arange(new_w) * w / new_w).astype(int), 0, w - 1)
    return img[np.ix_(ys, xs)]          # 花式索引一次性取子图，无 for 循环


nn_small = resize_nearest(G, 120, 120)
nn_back = resize_nearest(nn_small, G.shape[0], G.shape[1])
cv_nn = cv2.resize(np.clip(G, 0, 255).astype(np.uint8), (120, 120), interpolation=cv2.INTER_NEAREST)
print("手写最近邻 vs cv2.INTER_NEAREST 是否完全一致 :",
      bool(np.array_equal(nn_small, cv_nn)))
show_images([to_uint8(G), nn_back], titles=["原图", f"最近邻缩到120再放大回{G.shape[1]}（马赛克）"], cols=2)

In [ ]:
# ---------- 1.5.2b 双线性插值（纯 NumPy 向量化）----------
def resize_bilinear(img, new_h, new_w):
    """双线性插值缩放（向量化实现，灰度/彩色通用）。返回 float64。"""
    h, w = img.shape[:2]
    f = img.astype(np.float64)
    # 输出像素中心 -> 原图坐标
    ys = np.clip((np.arange(new_h) + 0.5) * h / new_h - 0.5, 0, h - 1)
    xs = np.clip((np.arange(new_w) + 0.5) * w / new_w - 0.5, 0, w - 1)
    y0 = np.floor(ys).astype(int)
    x0 = np.floor(xs).astype(int)
    y1 = np.minimum(y0 + 1, h - 1)
    x1 = np.minimum(x0 + 1, w - 1)
    wy = (ys - y0)
    wx = (xs - x0)
    if f.ndim == 3:                       # 彩色：给权重补一个通道维
        wy = wy[:, None, None]
        wx = wx[None, :, None]
    else:
        wy = wy[:, None]
        wx = wx[None, :]
    Ia = f[np.ix_(y0, x0)]
    Ib = f[np.ix_(y0, x1)]
    Ic = f[np.ix_(y1, x0)]
    Id = f[np.ix_(y1, x1)]
    top = Ia * (1 - wx) + Ib * wx         # 上边两点横向插值
    bot = Ic * (1 - wx) + Id * wx         # 下边两点横向插值
    return top * (1 - wy) + bot * wy      # 再纵向插值


bl_small = resize_bilinear(G, 120, 120)
bl_back = resize_bilinear(bl_small, G.shape[0], G.shape[1])
cv_bl = cv2.resize(np.clip(G, 0, 255).astype(np.uint8), (120, 120), interpolation=cv2.INTER_LINEAR)
diff = np.abs(bl_small - cv_bl.astype(np.float64))
print(f"手写双线性 vs cv2.INTER_LINEAR：最大差 {diff.max():.2f}，平均差 {diff.mean():.3f}")
print("（差异来自 cv2 内部的定点数舍入，<1 灰度级属正常）")
show_images([nn_back, to_uint8(bl_back)],
            titles=["最近邻放大（锯齿）", "双线性放大（平滑）"], cols=2,
            suptitle="两种插值放大回原尺寸的对比")

In [ ]:
# ---------- 1.5.2c 向量化 vs 双层 for 循环：性能对比 ----------
import time


def resize_bilinear_loops(img, new_h, new_w):
    """教学用：双层 for 循环版双线性（慢，仅用于对比）。"""
    h, w = img.shape[:2]
    f = img.astype(np.float64)
    out = np.zeros((new_h, new_w), np.float64)
    for i in range(new_h):
        y = max(0.0, min((i + 0.5) * h / new_h - 0.5, h - 1))
        y0 = int(np.floor(y)); y1 = min(y0 + 1, h - 1); wy = y - y0
        for j in range(new_w):
            x = max(0.0, min((j + 0.5) * w / new_w - 0.5, w - 1))
            x0 = int(np.floor(x)); x1 = min(x0 + 1, w - 1); wx = x - x0
            out[i, j] = ((1 - wy) * ((1 - wx) * f[y0, x0] + wx * f[y0, x1])
                         + wy * ((1 - wx) * f[y1, x0] + wx * f[y1, x1]))
    return out


t0 = time.perf_counter()
resize_bilinear(G, 240, 240)
t_vec = time.perf_counter() - t0
t0 = time.perf_counter()
resize_bilinear_loops(G, 240, 240)
t_loop = time.perf_counter() - t0
print(f"向量化版   : {t_vec * 1000:8.2f} ms")
print(f"for 循环版 : {t_loop * 1000:8.2f} ms")
print(f"加速比     : {t_loop / max(t_vec, 1e-9):.0f}×  ← 这就是向量化的意义")

**运行结果说明**：向量化版通常比纯 Python 双循环快 **几十到上百倍**，因为 NumPy 把循环下沉到了 C 层。写图像处理代码时，**先想能不能写成整数组运算**，实在不行再考虑循环（或用 Numba/Cython）。

### 1.5.3 图像锐化：Unsharp Masking（反锐化掩模）

锐化的目标是**把高频（边缘、细节）加回去**。最常用的方法：

**Unsharp Masking**：$g = f + k \cdot (f - \text{blur}(f))$，即「原图 + k×高通分量」。

高通分量 $h = f - \text{blur}(f)$ 只包含边缘和细节；把它按系数 $k$（amount）加回原图就得到锐化效果。

In [ ]:
# ---------- 1.5.3a 用 NumPy 实现 2D 相关 / 卷积 ----------
from numpy.lib.stride_tricks import sliding_window_view

# SciPy 的边界命名与 np.pad 的命名不完全一致，这里做映射，
# 保证手写实现与 scipy.ndimage 在相同 mode 下可逐像素对照。
#   scipy 'reflect' = 边缘像素重复一次  = np.pad 'symmetric'
#   scipy 'mirror'  = 边缘像素不重复    = np.pad 'reflect'
_PAD_MAP = {"reflect": "symmetric", "mirror": "reflect", "constant": "constant",
            "nearest": "edge", "wrap": "wrap"}


def correlate2d_np(img, kernel, pad_mode="reflect", cval=0.0):
    """纯 NumPy 实现 2D 相关（核不翻转）。仅支持单通道。

    思路：np.pad 补边 -> sliding_window_view 取出所有邻域窗口
          -> einsum 与核做加权求和。全程无 Python 像素循环。
    pad_mode 使用与 SciPy 相同的命名（见 _PAD_MAP）。
    """
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    np_mode = _PAD_MAP[pad_mode]
    pad_kw = {"constant_values": cval} if np_mode == "constant" else {}
    padded = np.pad(img.astype(np.float64), ((ph, ph), (pw, pw)),
                    mode=np_mode, **pad_kw)
    windows = sliding_window_view(padded, (kh, kw))      # (H, W, kh, kw)
    return np.einsum("ijkl,kl->ij", windows, kernel.astype(np.float64))


def convolve2d_np(img, kernel, pad_mode="reflect", cval=0.0):
    """纯 NumPy 实现 2D 卷积 = 用翻转 180° 的核做相关。"""
    return correlate2d_np(img, kernel[::-1, ::-1], pad_mode, cval)


# 与 SciPy 对照验证
sp_corr = ndimage.correlate(G, K_ASYM, mode="reflect")
my_corr = correlate2d_np(G, K_ASYM, pad_mode="reflect")
sp_conv = ndimage.convolve(G, K_ASYM, mode="reflect")
my_conv = convolve2d_np(G, K_ASYM, pad_mode="reflect")
print("手写相关 vs scipy.correlate 最大差 :", float(np.abs(my_corr - sp_corr).max()))
print("手写卷积 vs scipy.convolve 最大差 :", float(np.abs(my_conv - sp_conv).max()))

In [ ]:
# ---------- 1.5.3b Unsharp Masking ----------
def unsharp(img, sigma=2.0, amount=1.0):
    """Unsharp masking：g = f + amount * (f - gaussian_blur(f))。"""
    f = img.astype(np.float64)
    blur = ndimage.gaussian_filter(f, sigma=sigma)
    return np.clip(f + amount * (f - blur), 0, 255)


imgs, titles = [to_uint8(G)], ["原图"]
for amt in [0.5, 1.0, 2.0]:
    imgs.append(unsharp(G, sigma=2.0, amount=amt).astype(np.uint8))
    titles.append(f"amount={amt}")
show_images(imgs, titles=titles, cols=4, suptitle="Unsharp Masking：amount 越大越锐（也越容易出光晕）")

# 高通 = 原图 - 低通，单独看一下被"加回去"的是什么
highpass = G - ndimage.gaussian_filter(G, sigma=2.0)
show_images([to_uint8(np.clip(highpass + 128, 0, 255))],
            titles=["高通分量 (f - blur) + 128 偏置"], cols=1,
            suptitle="被加回的高频：几乎只有边缘")

**运行结果说明**：高通分量图（+128 偏置后中灰为背景）里**只剩边缘轮廓**，印证了「锐化 = 原图 + k×边缘」。`amount` 过大会在强边缘两侧产生**过冲光晕**（halo），实际使用需克制。

### 1.5.4 图像平均降噪

对同一场景的 $N$ 张独立同噪声图像求平均：
$$ \bar g = \frac1N \sum_{i=1}^N (f + n_i) = f + \frac1N \sum n_i $$
信号 $f$ 不变，噪声均值的标准差降为 $\sigma/\sqrt N$。即**张数翻 4 倍，噪声减半**。这是天文摄影、荧光显微、多帧 HDR 的基本原理。

In [ ]:
# ---------- 1.5.4a 多帧平均：N 越大越干净 ----------
clean_u8 = to_uint8(G)
frames = make_noisy_frames(clean_u8, n_frames=32, sigma=30, seed=500)
Ns = [1, 2, 4, 8, 16, 32]
imgs, titles = [], []
for N in Ns:
    avg = np.mean([f.astype(np.float64) for f in frames[:N]], axis=0)
    imgs.append(to_uint8(avg))
    titles.append(f"N={N}\nPSNR={psnr(clean_u8, avg):.1f}dB")
show_images(imgs, titles=titles, cols=6, suptitle="多帧平均降噪：噪声随 N 增大而衰减")

In [ ]:
# ---------- 1.5.4b 定量验证：噪声标准差 ~ σ/√N ----------
noise_std, theory, psnr_list = [], [], []
for N in range(1, 33):
    avg = np.mean([f.astype(np.float64) for f in frames[:N]], axis=0)
    resid = avg - clean_u8.astype(np.float64)
    noise_std.append(resid.std())
    theory.append(30.0 / np.sqrt(N))
    psnr_list.append(psnr(clean_u8, avg))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, 33), noise_std, "o-", label="实测残差标准差", color="#2E9BD6")
axes[0].plot(range(1, 33), theory, "--", label=r"理论 $\sigma/\sqrt{N}$", color="#E8A33D")
axes[0].set_xlabel("平均张数 N")
axes[0].set_ylabel("噪声标准差")
axes[0].set_title("噪声按 1/√N 衰减")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(range(1, 33), psnr_list, "s-", color="#27ae60")
axes[1].set_xlabel("平均张数 N")
axes[1].set_ylabel("PSNR (dB)")
axes[1].set_title("PSNR 随 N 单调上升（约 +3dB / 张数×2）")
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"N=1  噪声σ={noise_std[0]:.2f}  PSNR={psnr_list[0]:.2f} dB")
print(f"N=32 噪声σ={noise_std[31]:.2f}  PSNR={psnr_list[31]:.2f} dB")
print("经验：张数每 ×4，噪声标准差 ÷2，PSNR 约 +6 dB")

**运行结果说明**：实测残差标准差曲线与理论 $\sigma/\sqrt N$ 几乎重合，验证了多帧平均的统计性质。PSNR 每把张数翻倍约提升 3 dB（因为方差减半 ≈ 功率 -3 dB，PSNR 是功率比）。

> 📌 **§1.5 小结**
> | 算法 | 手写要点 |
> | :--- | :--- |
> | 灰度变换 | 先算 256 项 LUT，再 `lut[img]` 查表 |
> | 最近邻缩放 | 坐标反映射 + `np.ix_` 花式索引 |
> | 双线性缩放 | 2×2 邻域按距离加权，权重广播 |
> | 卷积/相关 | `pad` + `sliding_window_view` + `einsum` |
> | 锐化 | `f + k(f - blur)`（Unsharp Masking） |
> | 多帧平均 | `np.mean(frames, axis=0)`，噪声 ∝ 1/√N |

---

## 1.7 本课小结与练习

### 知识地图

| 库 | 一句话定位 | 本课关键 API |
| :--- | :--- | :--- |
| **NumPy** | 图像即数组，一切的地基 | 切片/广播/花式索引、`interp`、`sliding_window_view` |
| **OpenCV** | 工业级视觉算法库（BGR） | `imread/imwrite`、`cvtColor`、`resize`、绘图 |
| **Pillow** | 文件 I/O 与轻量编辑（RGB） | `Image.open/convert/resize/rotate/filter`、`ImageEnhance` |
| **SciPy** | 可自定义核的滤波/卷积 | `uniform_filter`、`gaussian_filter`、`median_filter`、`convolve`、`correlate` |

### 五条实践原则

1. **读图之后第一件事**：确认通道顺序（BGR vs RGB）与 dtype（uint8 vs float）；
2. **在 float 上运算，在 uint8 上显示**：中间过程保持浮点精度，输出前 `clip + astype`；
3. **先查库有没有现成 API**：OpenCV / SciPy 已高度优化，手写只为理解原理；
4. **向量化优先**：能用整数组运算就不要写 for 循环（性能差几十倍）；
5. **用 PSNR/MSE 做定量评估**：视觉对比只是第一步，数字才有说服力。

In [ ]:
# ---------- 1.7 本课产物清单 ----------
print("本课在 WORK_DIR 生成的文件：")
for fn in sorted(os.listdir(WORK_DIR)):
    fp = os.path.join(WORK_DIR, fn)
    print(f"  {fn:<36s} {os.path.getsize(fp) // 1024:>5d} KB")
print("\n下一站：打开 02_练习题.ipynb 动手实践")